# Geneation of features

In [131]:
import glob
import re
import itertools

import matplotlib.pyplot as plt
import scipy.signal
from scipy import stats, signal
import seaborn as sns
import numpy as np
import pandas as pd

In [132]:
import bidict

In [133]:
import pymc as pm

In [134]:
import arviz as az

In [135]:
import pytensor as pt

In [136]:
from scipy import io

In [137]:
import os

Idea: 
 - focus on a single user for now
  -  estimate parameters from the non-intensive activities time series
  - build a model to predict/impute values for the intensive activities  

- restrict to z axis?
- restrict to accelometer?

## Data etc

In [138]:
phone_acc_df = pd.read_pickle('data/phone_acc_df.pkl')
phone_gyro_df = pd.read_pickle('data/phone_gyro_df.pkl')
watch_acc_df = pd.read_pickle('data/watch_acc_df.pkl')
watch_gyro_df = pd.read_pickle('data/watch_gyro_df.pkl')

FileNotFoundError: [Errno 2] No such file or directory: 'data/phone_acc_df.pkl'

In [ ]:
activity_codes = bidict.bidict({
    'A': 'walking',
    'B': 'jogging',
    'C': 'stairs',
    'D': 'sitting',
    'E': 'standing',
    'F': 'typing',
    'G': 'teeth',
    'H': 'soup',
    'I': 'chips',
    'J': 'pasta',
    'K': 'drinking',
    'L': 'sandwich',
    'M': 'kicking',
    'O': 'catch',
    'P': 'dribbling',
    'Q': 'writing',
    'R': 'clapping',
    'S': 'folding'
})

In [ ]:
activity_groups = {
    "non-hand": ['walking', 'jogging', 'stairs', 'standing', "kicking"],
    "hand": ["dribbling","catch", "typing", "writing", "clapping",  "teeth", "folding"],
    "hand-eating": ["soup", "chips", "pasta", "sandwich", "drinking"]
}

In [ ]:
user_id = 1610

## Auxiliary features

In [ ]:
idx_eat = phone_acc_df.loc[: , 'ACTIVITY'].isin({activity_codes.inverse[desc] for desc in activity_groups['hand-eating']}) 
df_eat = phone_acc_df.loc[idx_eat]

In [ ]:
idx_act = phone_acc_df.loc[:, 'ACTIVITY'].isin({activity_codes.inverse[desc] for desc in activity_groups['hand']})
df_act = phone_acc_df.loc[idx_act]

### Means

In [ ]:
df_eat.reset_index()['RESULTANT'].plot()
df_act.reset_index()['RESULTANT'].plot()


In [ ]:
eat_avgs  = df_eat.drop(columns="ACTIVITY").reset_index().groupby("class").mean()

In [ ]:
act_avgs  = df_act.drop(columns="ACTIVITY").reset_index().groupby("class").mean()

`FIXME: what to do with the histogram features?`

In [ ]:
eat_avgs

### Peak means

In [ ]:
scipy.signal.find_peaks(df_eat["RESULTANT"], prominence=0.1)

In [ ]:
df_eat["RESULTANT"].iloc[scipy.signal.find_peaks(df_eat["RESULTANT"], prominence=0.1)[0]
]

In [ ]:
def find_peak_mean(column, prominence=0.1):
        return column.iloc[scipy.signal.find_peaks(column, prominence=prominence)[0]].mean()


In [ ]:
eat_peak_means = df_eat.drop(columns="ACTIVITY").reset_index().groupby("class").aggregate(lambda x: find_peak_mean(x, prominence=0.05))
# eat_peak_means.head()

In [ ]:
act_peak_means = df_act.drop(columns="ACTIVITY").reset_index().groupby("class").aggregate(lambda x: find_peak_mean(x, prominence=0.05))


### Save

In [ ]:
df_eat.to_parquet("data/df_eat.parquet")

In [ ]:
df_act.to_parquet("data/df_act.parquet")

In [ ]:
features = pd.concat([eat_avgs, eat_peak_means, act_avgs, act_peak_means],
                   keys=["Eat_avg", "Eat_peak", "Act_avg", "Act_peak"]
                  )

In [ ]:
features.to_parquet("data/features.parquet")

### Total minutes

In [ ]:
total_min = phone_acc_df.groupby(by=phone_acc_df.index)['ACTIVITY'].count() * 10 / 60
total_min

In [ ]:
total_min_eat = df_eat.groupby(by=df_eat.index)['ACTIVITY'].count() * 10 / 60
total_min_act = df_act.groupby(by=df_act.index)['ACTIVITY'].count() * 10 / 60

In [ ]:
total_min_eat_act = df_eat.groupby(by=[df_eat.index, "ACTIVITY"])['ACTIVITY'].count() * 10 / 60
total_min_eat_act

In [ ]:
total_min_act_act = df_act.groupby(by=[df_act.index, "ACTIVITY"])['ACTIVITY'].count() * 10 / 60 

In [ ]:
pd.concat([total_min, total_min_eat, total_min_act], keys=["Combined", "Eat", "Act"]).to_frame().to_parquet("data/total_min.parquet")

In [ ]:
pd.concat([total_min_eat_act, total_min_act_act], keys=["Eat", "Act"]).to_frame().to_parquet("data/total_min_detailed.parquet")

### Histogram derived-features

Problem: what the hell are the ranges here? They don't save them!

In [ ]:
df_act.filter(regex=r"X\d+").head()

Just assume $[0, 1]$ and calculate the entropies

In [ ]:
for ii, row in df_act.filter(regex=r"X\d+").head().iterrows():
    plt.figure()
    print(stats.entropy(row))
    row.plot.bar()
    plt.show()

In [ ]:
entropiesX = []
user_id = 1610
for ii, row in df_act.loc[user_id].filter(regex=r"X\d+").iterrows():
    entropiesX.append(stats.entropy(row))
    
plt.plot(entropiesX / np.sum(entropiesX), label="entropy")
# res = df_act.loc[user_id, "RESULTANT"].reset_index()
# plt.plot(res / res.sum(), label="resultant")
xavg = df_act.loc[user_id, "XAVG"].reset_index()
plt.plot(xavg / xavg.sum(), label="xavg")
plt.legend()

In [ ]:
act_entrX = df_act.filter(regex=r"X\d+").apply(stats.entropy, axis=1).groupby("class").mean()
act_entrY = df_act.filter(regex=r"Y\d+").apply(stats.entropy, axis=1).groupby("class").mean()
act_entrZ = df_act.filter(regex=r"Z\d+").apply(stats.entropy, axis=1).groupby("class").mean()

In [ ]:
eat_entrX = df_eat.filter(regex=r"X\d+").apply(stats.entropy, axis=1).groupby("class").mean()
eat_entrY = df_eat.filter(regex=r"Y\d+").apply(stats.entropy, axis=1).groupby("class").mean()
eat_entrZ = df_eat.filter(regex=r"Z\d+").apply(stats.entropy, axis=1).groupby("class").mean()

In [ ]:
entropies = pd.concat([act_entrX, act_entrY, act_entrZ, 1/3 * (act_entrX + act_entrY + act_entrZ),
           eat_entrX, eat_entrY, eat_entrZ, 1/3 * (eat_entrX + eat_entrY + eat_entrZ)],
          keys=["act_entrX", "act_entrY", "act_entrZ", "act_entr_mean", "eat_entrX", "eat_entrY", "eat_entrZ", "eat_entr_mean",])
entropies.to_frame().to_parquet("data/entropies.parquet")

### Spectral/temporal

In [ ]:
df_act.loc[user_id].reset_index()["RESULTANT"].plot()

In [ ]:
df_act.loc[user_id].reset_index()["XPEAK"].plot()

In [ ]:
# df_act.loc[user_id].reset_index()["YPEAK"].plot()
# df_act.loc[user_id].reset_index()["ZPEAK"].plot()

MFCC

In [ ]:
df_act.loc[user_id].reset_index().filter(regex=r"XMFCC\d+").plot()

In [ ]:
df_act.loc[user_id].reset_index()["RESULTANT"].plot()


Seems again a proxy for the negative of the resultant

**Cosine similarity**

In [ ]:
df_act.loc[user_id].reset_index()["XZCOS"].plot()
df_act.loc[user_id].reset_index()["XYCOS"].plot()
df_act.loc[user_id].reset_index()["YZCOS"].plot()
plt.legend()

In [ ]:
df_act.loc[user_id].reset_index()["XZCOR"].plot()
df_act.loc[user_id].reset_index()["XYCOR"].plot()
df_act.loc[user_id].reset_index()["YZCOR"].plot()
plt.legend()